# World Model from Scratch

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

### (V) Vision Model 

It compresses information, i.e. it *encodes the high dimensional observation* \[ the input \] *into a low dimensional latent vector*. The paper uses a **Variational AutoEncoder** for this purpose.

Takes the original input.


### Variational AutoEncoder

Takes a vector as input, which is passed through two separate CNNs. The first one gives $\mu$ (the center/mean of the distribution), while the second one gives $\sigma$ (the spread of the distribution). Thus, the two CNNs point to an area in a latent vector space, which theoretically contains abstract representations of concepts.

The fact that it is an area instead of a concrete point gives the VAE the ability to generate novel images, that have not been encountered during training.

Using $\mu$ and $\sigma$, we compute $z = \mu + \sigma * \epsilon$, where $\epsilon$ is random noise drawn from $N(0, 1)$, which is a fixed distribution outside of the model. Thus, the randomness is in $\epsilon$, which is a constant, allowing us to backpropagate.

In practice, we do:

```python
mu = hidden @ W_mu + b_mu
log_sigma = hidden @ W_sigma + b_sigma
sigma = exp(log_sigma)
```

So that sigma is always positive. 

Then, the decoder is a mirror of the encoder. 

In [2]:
class VariationalAutoEncoder(nn.Module):
    def __init__(self, d_latent):
        super().__init__()
        self.d_latent = d_latent
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(4096, d_latent)
        self.fc_log_sigma = nn.Linear(4096, d_latent)
        self.fc_decoder = nn.Linear(d_latent, 4096)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            #sigmoid corresponds to normalized pixel values
            nn.Sigmoid()
        )

    def forward(self, x):
        # [batch_size, 3, 64, 64] => [batch_size, 256,  4,  4]
        enc = self.encoder(x)
    
        # [batch_size, 256,  4,  4] => [batch_size, 4096]
        enc = enc.flatten(start_dim=1) 

        # [batch_size, 4096] => [batch_size, d_latent]
        mu = self.fc_mu(enc)

        # [batch_size, 4096] => [batch_size, d_latent]
        log_sigma = self.fc_log_sigma(enc)
        sigma = torch.exp(log_sigma)

        # sample epsilon from N(0, 1)
        epsilon = torch.randn_like(sigma)
        
        z = mu + sigma * epsilon

        # [batch_size, d_latent] -> [batch_size, 4096]
        dec_in = self.fc_decoder(z)

        # [batch_size, 4096] -> [batch_size, 256, 4, 4]
        dec_in = dec_in.reshape(-1, 256, 4, 4) 

        # [batch_size, 256, 4, 4] -> [batch_size, 3, 64, 64]]
        reconstruction = self.decoder(dec_in)

        return reconstruction, mu, log_sigma, z

For loss, we use MSE alongside KL divergence, which computes the difference between the two Gaussian distributions, $N(\mu, \sigma)$ and $N(0,1)$, the distribution we sample $\epsilon$ from. We do this because we want the latent space to resemble $N(0,1)$, so that the latent space is smooth and continuous (if there are discontinuities in it, then the decoder will have blind spots).

$$\text{Loss} = ||\text{input} - \text{reconstruction}||² + KL(N(\mu, \sigma) || N(0,1))$$

The KL formula is self-correcting:
- μ drifts from 0 → KL penalizes it
- σ drifts from 1 → KL penalizes it

$$KL = -0.5 * \sum(1 + log(\sigma^2) - \mu^2 - \sigma^2)$$

In [3]:
def vae_loss(input, reconstruction, mu, log_sigma):
    loss = F.mse_loss(input, reconstruction) - 0.5 * torch.sum(1 + 2 * log_sigma - mu**2 - torch.exp(2*log_sigma))
    return loss

### (M) Memory RNN
Uses past states (*historical codes to create a representation*) to predict future states. Takes **(V)**'s output as input.

While **(V)** sees and compresses what the agent sees, **(M)** adds a time dimension, so to speak. Its purpose is to predict future $z$ vectors that **(V)** will produce. 

*Since many complex environments are stochastic in nature, we train our RNN to output a probability density function p(z) instead of a deterministic prediction of z.*

Stochastic environment means unexpected things can happen, so the model needs something that allows more wiggle room, i.e. a probability distribution. A simple Gaussian distribution doesn't cut it, because, for instance, say there are two monsters, each with an equal chance of shooting the agent (who needs to dodge). Then, the model wouldn't know where to center the gaussian, it would need two "humps". We thus need a multinomial distribution.

The **(M)** model needs to output the parameters of this mix of Gaussian distributions:
1. Mean
2. Variance
3. Weight (i.e. how probable is this particular Gaussian compared to the others; all weights must sum to 1)
All for each dimension of $z$.

The RNN will model $P(z_{t+1} | a_t, z_t, h_t)$, where $a_t$ is the just taken action and $h_t$ embeds the history (what happened before). Note that $h_t$ is embedded in the RNN's architecture, it's not an explicit parameter; the RNN takes as input a concatenation of $a_t$ and $z_t$.

Then, the hidden state of the RNN $h_t$ is fed into the MDN (Mixture Density Network), which, in turn, produces the aforementioned multinomial distribution.

MDN is just a single linear layer that takes $h_t$ as input and outputs a $[\mu + \sigma + \pi]$:
- $\mu: \text{num\_gaussians} \times \text{d\_latent}$
- $\sigma: \text{num\_gaussians} \times \text{d\_latent}$
- $\pi: \text{num\_gaussians}$

Then:
- reshape and split that output into $\mu$, $\sigma$, and $\pi$.
- apply exp to $\log{\sigma}$ to get $\sigma > 0$ 
- apply softmax to $\pi$ to get weights that sum to 1
- $\mu$ has no activation, it can be any real number

In the end, to compute z, we do:
$$z = \mu + \tau * \sigma * \epsilon$$
Where $\tau$ is the temperature, i.e. a scalar which modifies the sampling region, so:
- Low $\tau$ (e.g. 0.1): $\sigma$ shrinks => samples cluster tightly around $\mu$ => more predictable environment
- High $\tau$ (e.g. 1.15): $\sigma$ grows → samples spread out → more random, chaotic environment


The flow is:
$$z_t, a_t \rightarrow RNN \rightarrow h_t \rightarrow \text{MDN head} \rightarrow (\pi, \mu, \sigma) → \text{sample} \ z_{t+1}$$

In [4]:
class Model(nn.Module):
    def __init__(self, d_latent, d_hidden, num_gaussians, num_actions):
        super().__init__()
        self.d_latent = d_latent
        self.d_hidden = d_hidden
        self.num_gaussians = num_gaussians
        self.num_actions = num_actions

        # |{Gas, Brake, Turn (L or R)}| + d_latent
        self.rnn = nn.LSTM(d_latent + num_actions, d_hidden)
        self.fc_mdn = nn.Linear(d_hidden, 2 * num_gaussians * d_latent + num_gaussians)

    def forward(self, z_t, a_t, h):
        x = torch.cat([z_t, a_t], dim=-1)
        x = x.view(1, 1, -1)
        _, (h_t, c_t) = self.rnn(x, h)

        # [1, 256] -> [256]
        h_t = h_t.squeeze(0)
        c_t = c_t.squeeze(0)
        
        out = self.fc_mdn(h_t)
        log_sigma = out[:, :self.num_gaussians * self.d_latent]
        sigma = torch.exp(log_sigma)
        mu = out[:, self.num_gaussians * self.d_latent : 2 * self.num_gaussians * self.d_latent] 
        pi = out[:, 2 * self.num_gaussians * self.d_latent : ]
        pi = F.softmax(pi, dim=-1)

        return mu, sigma, pi, (h_t, c_t)

### (C) Controller
**(C)** determines the correct course of action given $h_t$ and $z_t$. It is a simple linear layer that maps them to the vector of potential actions

In [5]:
class Controller(nn.Module):
    def __init__(self, d_hidden, d_latent, num_actions):
        super().__init__()
        self.W_c = nn.Linear(d_hidden + d_latent, num_actions)

    def forward(self, z_t, h_t):
        x = torch.cat([z_t, h_t], dim=-1)
        a_t = self.W_c(x)
        a_t = F.tanh(a_t)

        return a_t

### Putting it all together

In [6]:
class WorldModel(nn.Module):
    def __init__(self, d_latent, d_hidden, num_gaussians, num_actions, temperature, environment):
        super().__init__()
        self.d_latent = d_latent
        self.d_hidden = d_hidden
        self.num_gaussians = num_gaussians
        self.num_actions = num_actions
        self.temperature = temperature
        self.environment = environment

        self.V = VariationalAutoEncoder(d_latent)
        self.M = Model(d_latent, d_hidden, num_gaussians, num_actions)
        self.C = Controller(d_hidden, d_latent, num_actions)

    def rollout(self):
        obs = self.environment.reset()
        h = (torch.zeros(1, 1, self.d_hidden), torch.zeros(1, 1, self.d_hidden))
        done = False
        cumulative_reward = 0

        while not done:
            obs_tensor = torch.FloatTensor(obs).permute(2, 0, 1).unsqueeze(0)

            with torch.no_grad:
                reconstruction, mu_v, log_sigma_v, z = self.V(obs_tensor)
                
                a = self.C(z, h[0].squeeze(0))

                obs, reward, done = self.environment.step(a.squeeze(0).detach().numpy())
                cumulative_reward += reward

                _, _, _, h = self.M(z, a, h)

        return cumulative_reward
